# Quaterback Support

In [1]:
import numpy as np
import nflreadpy as nfl
import plotly.io as pio
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import polars as pl
import pandas as pd
import json
from scipy.special import erf

In [2]:
pbp = nfl.load_pbp(seasons=[2025])
ftn = nfl.load_ftn_charting(seasons=[2025])
nextgen_passing = nfl.load_nextgen_stats(seasons=[2025], stat_type="passing")
nextgen_passing = nextgen_passing.filter(pl.col("season_type") == "REG").to_pandas()
nextgen_rushing = nfl.load_nextgen_stats(seasons=[2025], stat_type="rushing")
nextgen_rushing = nextgen_rushing.filter(pl.col("season_type") == "REG")
nextgen_receiving = nfl.load_nextgen_stats(seasons=[2025], stat_type="receiving")
nextgen_receiving = nextgen_receiving.filter(pl.col("season_type") == "REG")

teams = nfl.load_teams()
players = nfl.load_players().to_pandas()
teams = teams.select(["team_id", "team_abbr", "team_color", "team_logo_espn"])
teams_pd = teams.to_pandas()

teams_pd["team_id"] = teams_pd["team_id"].astype(str)

left_keys  = ["game_id", "play_id"]
right_keys = ["nflverse_game_id", "nflverse_play_id"]

pbp2 = pbp.with_columns([
    pl.col("play_id").cast(pl.Int64),
])

ftn2 = ftn.with_columns([
    pl.col("nflverse_play_id").cast(pl.Int64),
])

ftn_clean = ftn2.select(
    right_keys + [c for c in ftn2.columns if c not in pbp2.columns and c not in right_keys]
)

pbp = (
    pbp2
    .join(
        ftn_clean,
        left_on=left_keys,
        right_on=right_keys,
        how="left"
    )
)


In [3]:
with open("data/2025_ngs_team_passing_offense.json", "r") as f:
    data = json.load(f)

ngs_passing_offense = pd.json_normalize(data["offense"])

ngs_passing_offense_filtered = ngs_passing_offense[["teamId", "qbpPct", "ttp"]]

ngs_passing_offense_filtered = ngs_passing_offense_filtered.merge(
    teams_pd[["team_id", "team_abbr"]],
    left_on="teamId",
    right_on="team_id",
    how="left"
).drop(columns=["team_id"])

ngs_passing_offense_filtered = ngs_passing_offense_filtered.sort_values("qbpPct", ascending=False)

ngs_passing_offense_filtered = ngs_passing_offense_filtered.drop(ngs_passing_offense_filtered[ngs_passing_offense_filtered["team_abbr"].isin(["SD", "LAR", "STL", "OAK"])].index)

print(ngs_passing_offense_filtered.columns)
print(ngs_passing_offense_filtered.shape)
ngs_passing_offense_filtered.head(36)

Index(['teamId', 'qbpPct', 'ttp', 'team_abbr'], dtype='object')
(32, 4)


,teamId,qbpPct,ttp,team_abbr
34,1050,0.46479,2.73794,CLE
19,4400,0.42464,2.63321,LAC
35,3430,0.39869,2.55909,NYJ
8,3800,0.38688,2.76838,ARI
32,3000,0.38394,2.63768,MIN
5,3200,0.38336,2.84286,NE
23,3410,0.37778,2.90955,NYG
28,0750,0.37500,2.59227,NaN
33,2100,0.36615,2.86606,TEN
30,2520,0.36047,2.54778,LV


In [4]:
empty_team = ngs_passing_offense_filtered[ngs_passing_offense_filtered["team_abbr"].isna()]

empty_team

,teamId,qbpPct,ttp,team_abbr
28,0750,0.37500,2.59227,NaN
21,0200,0.35385,2.59104,NaN
29,0325,0.33659,2.84774,NaN
7,0920,0.33188,2.60045,NaN
11,0810,0.31318,2.87784,NaN
16,0610,0.28401,2.90069,NaN


In [5]:
ngs_passing_offense_filtered.loc[ngs_passing_offense_filtered["teamId"] == "0750", ["team_abbr"]] = "CAR"
ngs_passing_offense_filtered.loc[ngs_passing_offense_filtered["teamId"] == "0200", ["team_abbr"]] = "ATL"
ngs_passing_offense_filtered.loc[ngs_passing_offense_filtered["teamId"] == "0325", ["team_abbr"]] = "BAL"
ngs_passing_offense_filtered.loc[ngs_passing_offense_filtered["teamId"] == "0920", ["team_abbr"]] = "CIN"
ngs_passing_offense_filtered.loc[ngs_passing_offense_filtered["teamId"] == "0810", ["team_abbr"]] = "CHI"
ngs_passing_offense_filtered.loc[ngs_passing_offense_filtered["teamId"] == "0610", ["team_abbr"]] = "BUF"

ngs_passing_offense_filtered.head(36)

,teamId,qbpPct,ttp,team_abbr
34,1050,0.46479,2.73794,CLE
19,4400,0.42464,2.63321,LAC
35,3430,0.39869,2.55909,NYJ
8,3800,0.38688,2.76838,ARI
32,3000,0.38394,2.63768,MIN
5,3200,0.38336,2.84286,NE
23,3410,0.37778,2.90955,NYG
28,0750,0.37500,2.59227,CAR
33,2100,0.36615,2.86606,TEN
30,2520,0.36047,2.54778,LV


In [6]:
dupes = ngs_passing_offense_filtered[ngs_passing_offense_filtered.duplicated(subset=["team_abbr"], keep=False)]

dupes

,teamId,qbpPct,ttp,team_abbr


In [7]:
ngs_qbs_filtered = nextgen_passing[["player_short_name", "player_gsis_id", "team_abbr", "attempts", "passer_rating", "completion_percentage_above_expectation"]]

ngs_qbs_filtered = (
    ngs_qbs_filtered
    .merge(
        players[["gsis_id", "headshot"]],
        left_on="player_gsis_id",
        right_on="gsis_id",
        how="left"
    )
    .drop(columns=["gsis_id"])
)

ngs_qbs_filtered = (
    ngs_qbs_filtered
    .sort_values("attempts", ascending=False)
    .head(40)
)

print(ngs_qbs_filtered.columns)
ngs_qbs_filtered.head()

Index(['player_short_name', 'player_gsis_id', 'team_abbr', 'attempts',
       'passer_rating', 'completion_percentage_above_expectation', 'headshot'],
      dtype='object')


,player_short_name,player_gsis_id,team_abbr,attempts,passer_rating,completion_percentage_above_expectation,headshot
29,B.Nix,00-0039732,DEN,612,87.806373,-2.067418,https://static.www.nfl.com/image/upload/f_auto...
38,D.Prescott,00-0033077,DAL,600,99.527778,4.445367,https://static.www.nfl.com/image/upload/f_auto...
1,M.Stafford,00-0026498,LAR,597,109.195282,1.476147,https://static.www.nfl.com/image/upload/f_auto...
24,J.Goff,00-0033106,DET,578,105.485871,1.581453,https://static.www.nfl.com/image/upload/f_auto...
0,C.Williams,00-0039918,CHI,568,90.126174,-6.874683,https://static.www.nfl.com/image/upload/f_auto...


In [8]:
with open("data/2025_ngs_receiving.json", "r") as f:
    data = json.load(f)

ngs_receivers = pd.json_normalize(data["receivers"])

ngs_receivers_filtered = ngs_receivers[["teamId", "tgt", "xCatch", "rec"]]

ngs_receivers_filtered["xrec"] = (ngs_receivers_filtered["xCatch"] * ngs_receivers_filtered["tgt"]).astype(int)

ngs_receivers_filtered = ngs_receivers_filtered.drop(columns=["xCatch"])

print(ngs_receivers_filtered.shape)
ngs_receivers_filtered.head()

(565, 4)


/var/folders/_v/f9jlhvnd2xg9kybwfc9yh6nw0000gn/T/ipykernel_68660/1879565970.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ngs_receivers_filtered["xrec"] = (ngs_receivers_filtered["xCatch"] * ngs_receivers_filtered["tgt"]).astype(int)


,teamId,tgt,rec,xrec
0,3800,169,126,114
1,3800,126,78,72
2,0920,185,125,124
3,1400,124,74,69
4,1200,137,93,81


In [9]:
def grade_metric(df, col, high_is_good=True):
    mean = df[col].mean()
    std = df[col].std()

    z = (df[col] - mean) / std

    if not high_is_good:
        z = -z

    z = z.clip(-3, 3)

    grade = (0.5 * (1 + erf(z / np.sqrt(2)))) * 100

    return grade

## Pass Protction

In [10]:
sack_not_qb = (
    pbp
    .filter(
        (pl.col("play_type") == "pass") &
        (pl.col("sack") == 1) &
        (pl.col("is_qb_fault_sack") == False)
    )
    .group_by("posteam")
    .agg(pl.count("play_id").alias("sack_not_qb"))
)
    
sack_not_qb = sack_not_qb.to_pandas()
sack_not_qb.head()

,posteam,sack_not_qb
0,JAX,30
1,CAR,25
2,WAS,20
3,DEN,16
4,TEN,35


In [11]:
pass_pro = (
    sack_not_qb
    .merge(
        ngs_passing_offense_filtered[["team_abbr", "qbpPct", "ttp"]],
        left_on="posteam",
        right_on="team_abbr",
        how="left"
    )
    .drop(columns=["team_abbr"])
)

print(pass_pro.shape)
pass_pro.head()

(32, 4)


,posteam,sack_not_qb,qbpPct,ttp
0,JAX,30,0.33282,2.69337
1,CAR,25,0.37500,2.59227
2,WAS,20,0.32753,2.86752
3,DEN,16,0.27720,2.74148
4,TEN,35,0.36615,2.86606


In [12]:
empty_sacks = pass_pro[pass_pro["sack_not_qb"].isna()]

empty_sacks

,posteam,sack_not_qb,qbpPct,ttp


In [13]:
pass_pro["pressure_rate_grade"] = grade_metric(pass_pro, "qbpPct", high_is_good=False)
pass_pro["time_to_pressure_grade"] = grade_metric(pass_pro, "ttp", high_is_good=True)
pass_pro["sack_grade"] = grade_metric(pass_pro, "sack_not_qb", high_is_good=False)

pass_pro["pass_pro_grade"] = (pass_pro["pressure_rate_grade"] + pass_pro["time_to_pressure_grade"] + pass_pro["sack_grade"]) / 3

pass_pro = pass_pro.sort_values("pass_pro_grade", ascending=False)

pass_pro.head()

,posteam,sack_not_qb,qbpPct,ttp,pressure_rate_grade,time_to_pressure_grade,sack_grade,pass_pro_grade
11,PIT,14,0.22559,2.81395,99.267577,75.872399,92.664013,89.267997
24,BUF,23,0.28401,2.90069,88.360296,90.947694,68.411561,82.573184
23,CHI,18,0.31318,2.87784,71.571711,87.903443,84.595218,81.356790
3,DEN,16,0.27720,2.74148,90.965664,56.804109,89.162587,78.977454
2,WAS,20,0.32753,2.86752,60.398313,86.315042,78.907943,75.207100


## Play Calling

In [14]:
easy_button = (
    pbp
    .filter(pl.col("play_type").is_in(["run", "pass"]))
    .filter(pl.col("epa").is_not_null())
    .with_columns([
        (
            (
                (pl.col("is_rpo") == "true") |
                (pl.col("is_screen_pass") == "true") |
                (pl.col("is_play_action") == "true")
            )
        ).cast(pl.Int8).alias("easy_button_play")
    ])
    .group_by("posteam")
    .agg([
        pl.mean("easy_button_play").round(3).alias("easy_button_rate"),
    ])
    .sort("easy_button_rate", descending=True)
).to_pandas()

easy_button.head()

,posteam,easy_button_rate
0,NYG,0.272
1,KC,0.269
2,IND,0.267
3,DEN,0.266
4,TEN,0.266


In [15]:
motion = (
    pbp
    .filter(pl.col("play_type").is_in(["run", "pass"]))
    .filter(pl.col("epa").is_not_null())
    .with_columns([
        (pl.col("is_motion") == True).cast(pl.Int8).alias("motion_play")
    ])
    .group_by("posteam")
    .agg([
        pl.mean("motion_play").round(3).alias("motion_rate"),
    ])
    .sort("motion_rate", descending=True)
).to_pandas()

motion.head()

,posteam,motion_rate
0,MIA,0.706
1,SF,0.674
2,ATL,0.668
3,LA,0.635
4,NYJ,0.629


In [16]:
play_calling = (
    motion.merge(easy_button, on="posteam", how="left")
)

play_calling.head()

,posteam,motion_rate,easy_button_rate
0,MIA,0.706,0.249
1,SF,0.674,0.162
2,ATL,0.668,0.199
3,LA,0.635,0.214
4,NYJ,0.629,0.229


In [17]:
play_calling["easy_button_grade"] = grade_metric(play_calling, "easy_button_rate", high_is_good=True)
play_calling["motion_grade"] = grade_metric(play_calling, "motion_rate", high_is_good=True)

play_calling["play_calling_grade"] = (play_calling["easy_button_grade"] + play_calling["motion_grade"]) / 2

play_calling = play_calling.sort_values("play_calling_grade", ascending=False)

play_calling.head()

,posteam,motion_rate,easy_button_rate,easy_button_grade,motion_grade,play_calling_grade
0,MIA,0.706,0.249,76.032495,98.055581,87.044038
7,BUF,0.615,0.245,72.094594,80.174958,76.134776
11,CHI,0.587,0.256,82.129546,68.199265,75.164406
4,NYJ,0.629,0.229,53.938189,84.971027,69.454608
15,GB,0.550,0.264,87.772758,49.132740,68.452749


## Pass Catching

In [18]:
seperation = (
    nextgen_receiving
    .group_by("team_abbr")
    .agg([
        pl.mean("avg_separation").alias("team_avg_separation"),
    ])
).to_pandas()

seperation.head()

,team_abbr,team_avg_separation
0,GB,2.795584
1,NYJ,3.045830
2,LAR,2.814450
3,HOU,2.831965
4,TB,2.899207


In [19]:
drop_rate = (
    pbp
    .filter(pl.col("pass_attempt") == 1)
    .group_by("posteam")
    .agg(
        (
            pl.col("is_drop").sum() /
            pl.col("is_catchable_ball").sum()
        ).alias("drop_rate")
    )
).to_pandas()

drop_rate.head()

,posteam,drop_rate
0,GB,0.056848
1,DET,0.056471
2,BAL,0.076923
3,CAR,0.062016
4,KC,0.061425


In [20]:
croe = (
    ngs_receivers_filtered
    .groupby("teamId")[["tgt","rec", "xrec"]]
    .sum()
    .reset_index()
)

croe["catch_rate"] = croe["rec"] / croe["tgt"]
croe["x_catch_rate"] = croe["xrec"] / croe["tgt"]
croe["catch_rate_over_expected"] = croe["catch_rate"] - croe["x_catch_rate"]

croe = croe.merge(
    teams_pd[["team_id", "team_abbr"]],
    left_on="teamId",
    right_on="team_id",
    how="left"
).drop(columns=["team_id"])

croe = croe.drop(croe[croe["team_abbr"].isin(["SD", "LAR", "STL", "OAK"])].index)

croe.loc[croe["teamId"] == "0750", ["team_abbr"]] = "CAR"
croe.loc[croe["teamId"] == "0200", ["team_abbr"]] = "ATL"
croe.loc[croe["teamId"] == "0325", ["team_abbr"]] = "BAL"
croe.loc[croe["teamId"] == "0920", ["team_abbr"]] = "CIN"
croe.loc[croe["teamId"] == "0810", ["team_abbr"]] = "CHI"
croe.loc[croe["teamId"] == "0610", ["team_abbr"]] = "BUF"

croe = croe.drop(columns=["teamId"])

croe.head()

,tgt,rec,xrec,catch_rate,x_catch_rate,catch_rate_over_expected,team_abbr
0,504,326,331,0.646825,0.656746,-0.009921,ATL
1,405,277,270,0.683951,0.666667,0.017284,BAL
2,504,363,342,0.720238,0.678571,0.041667,BUF
3,480,329,320,0.685417,0.666667,0.018750,CAR
4,532,334,366,0.627820,0.687970,-0.060150,CHI


In [21]:
yac = (
    pbp
    .group_by("posteam")
    .agg(pl.mean("yards_after_catch").alias("avg_yac"))
).to_pandas()

yac.head()

,posteam,avg_yac
0,None,NaN
1,GB,5.190341
2,LV,5.311765
3,SEA,5.461942
4,SF,4.633641


In [22]:
pass_catching = (
    drop_rate
    .merge(
        croe[["team_abbr", "catch_rate_over_expected"]], 
        left_on="posteam", 
        right_on="team_abbr",
        how="left")
    .drop(columns=["team_abbr"])
    .merge(
        seperation, 
        left_on="posteam", 
        right_on="team_abbr", 
        how="left")
    .drop(columns=["team_abbr"])
    .merge(
        yac, 
        on="posteam", 
        how="left"
    )
)

pass_catching.head()

,posteam,drop_rate,catch_rate_over_expected,team_avg_separation,avg_yac
0,GB,0.056848,0.060870,2.795584,5.190341
1,DET,0.056471,0.029091,3.199058,6.119289
2,BAL,0.076923,0.017284,3.364162,4.895683
3,CAR,0.062016,0.018750,2.671380,5.011429
4,KC,0.061425,-0.025547,3.323162,5.812672


In [23]:
pass_catching["drop_rate_grade"] = grade_metric(pass_catching, "drop_rate", high_is_good=False)
pass_catching["catch_rate_over_expected_grade"] = grade_metric(pass_catching, "catch_rate_over_expected", high_is_good=True)
pass_catching["separation_grade"] = grade_metric(pass_catching, "team_avg_separation", high_is_good=True)
pass_catching["yac_grade"] = grade_metric(pass_catching, "avg_yac", high_is_good=True)

pass_catching["pass_catching_grade"] = (
    pass_catching[["drop_rate_grade", "catch_rate_over_expected_grade", "separation_grade", "yac_grade"]]
    .mean(axis=1)
)

pass_catching = pass_catching.sort_values("pass_catching_grade", ascending=False)

pass_catching.head()

,posteam,drop_rate,catch_rate_over_expected,team_avg_separation,avg_yac,drop_rate_grade,catch_rate_over_expected_grade,separation_grade,yac_grade,pass_catching_grade
13,SEA,0.039409,0.057471,3.174221,5.461942,91.301243,87.861249,74.491547,73.346675,81.750178
22,BUF,0.063348,0.041667,3.479611,5.788413,37.116874,76.066530,97.519859,89.694232,75.099374
1,DET,0.056471,0.029091,3.199058,6.119289,56.209738,63.409245,77.777209,97.218716,73.653727
7,MIA,0.052023,0.021692,3.207940,5.590062,68.080102,55.074104,78.890923,80.918048,70.740795
8,NE,0.041304,0.112266,2.976531,5.109049,88.987532,99.712441,42.619423,47.228235,69.636908


## Run Game

In [24]:
rush_sr = (
    pbp
    .filter(
        (pl.col("play_type") == "run") &
        (pl.col("qb_scramble") == 0) 
        )
    .group_by("posteam")
    .agg(
        (pl.mean("success")).alias("rush_sr")
    )
).to_pandas()

rush_sr.head()

,posteam,rush_sr
0,CAR,0.420930
1,CIN,0.452514
2,GB,0.438178
3,NE,0.375502
4,WAS,0.428224


In [25]:
rush_epa = (
    pbp
    .filter(
        (pl.col("play_type") == "run") &
        (pl.col("qb_scramble") == 0) 
        )
    .group_by("posteam")
    .agg(
        (pl.mean("epa")).alias("rush_epa")
    )
).to_pandas()

rush_epa.head()

,posteam,rush_epa
0,PHI,-0.036965
1,LAC,-0.072477
2,DET,-0.049127
3,IND,0.066621
4,CHI,0.002676


In [26]:
rush_ypcoe = (
    nextgen_rushing
    .group_by("team_abbr")
    .agg([pl.mean("rush_yards_over_expected_per_att").alias("rush_ypcoe")])
).to_pandas()

rush_ypcoe.loc[rush_ypcoe["team_abbr"] == "LAR", ["team_abbr"]] = "LA"

rush_ypcoe.head()

,team_abbr,rush_ypcoe
0,MIA,0.415598
1,WAS,0.830207
2,NYJ,0.453602
3,SEA,0.342880
4,DEN,0.219764


In [27]:
run_game = (
    rush_sr
    .merge(rush_epa, on="posteam", how="left")
    .merge(rush_ypcoe, left_on="posteam", right_on="team_abbr", how="left")
    .drop(columns=["team_abbr"])
)

run_game.head()

,posteam,rush_sr,rush_epa,rush_ypcoe
0,CAR,0.420930,-0.056139,0.099533
1,CIN,0.452514,0.014184,0.346119
2,GB,0.438178,-0.066720,-0.016411
3,NE,0.375502,-0.090963,0.916637
4,WAS,0.428224,-0.036947,0.830207


In [28]:
run_game["rush_sr_grade"] = grade_metric(run_game, "rush_sr", high_is_good=True)
run_game["rush_epa_grade"] = grade_metric(run_game, "rush_epa", high_is_good=True)
run_game["rush_ypcoe_grade"] = grade_metric(run_game, "rush_ypcoe", high_is_good=True)

run_game["run_game_grade"] = (
    run_game[["rush_sr_grade", "rush_epa_grade", "rush_ypcoe_grade"]]
    .mean(axis=1)
)

run_game = run_game.sort_values("run_game_grade", ascending=False)

run_game.head()

,posteam,rush_sr,rush_epa,rush_ypcoe,rush_sr_grade,rush_epa_grade,rush_ypcoe_grade,run_game_grade
7,BUF,0.480000,0.069211,1.181998,96.451405,95.807343,97.095436,96.451395
24,LA,0.496078,0.036411,0.697312,98.748180,89.593904,78.484042,88.942042
6,BAL,0.423841,0.051947,1.170848,61.242420,93.074590,96.922735,83.746581
20,IND,0.450739,0.066621,0.513206,84.462944,95.464224,64.371262,81.432810
12,PIT,0.442500,-0.001969,0.875692,78.543208,76.076544,88.411703,81.010485


## Defense and Special Teams

In [29]:
def_epa = (
    pbp
    .filter(pl.col("play_type").is_in(["run", "pass"]))
    .group_by("defteam")
    .agg(pl.mean("epa").alias("def_epa"))
).to_pandas()

def_epa.head()

,defteam,def_epa
0,BUF,-0.010997
1,DAL,0.167388
2,TB,0.013279
3,CAR,0.070432
4,JAX,-0.086168


In [30]:
special_teams_epa_off = (
    pbp
    .filter(pl.col("special") == 1)
    .group_by("posteam")
    .agg(pl.sum("epa").alias("special_teams_epa_off"))
)

special_teams_epa_def = (
    pbp
    .filter(pl.col("special") == 1)
    .group_by("defteam")
    .agg(-pl.sum("epa").alias("special_teams_epa_def"))
)

special_teams_epa = (
    special_teams_epa_off
    .join(
        special_teams_epa_def,
        left_on="posteam",
        right_on="defteam",
        how="inner"
    )
).to_pandas()

special_teams_epa["special_teams_epa"] = special_teams_epa["special_teams_epa_off"] + special_teams_epa["special_teams_epa_def"]

special_teams_epa = special_teams_epa.drop(columns=["special_teams_epa_off", "special_teams_epa_def"])

special_teams_epa.head()

,posteam,special_teams_epa
0,HOU,51.777008
1,TB,-25.591023
2,ARI,-48.636138
3,NYJ,46.133113
4,LAC,-16.466128


In [31]:
avg_start = (
    pbp
    .filter(
        pl.col("play_type").is_in(["run", "pass"]) &
        (pl.col("play_id") == pl.col("drive_play_id_started"))
    )
    .group_by("posteam")
    .agg(pl.mean("yardline_100").alias("avg_start"))
).to_pandas()

avg_start.head()

,posteam,avg_start
0,JAX,66.914286
1,MIN,69.067416
2,CIN,70.310811
3,GB,70.961538
4,DEN,70.383929


In [32]:
def_and_st = (
    def_epa
    .merge(
        special_teams_epa[["posteam", "special_teams_epa"]],
        left_on="defteam",
        right_on="posteam",
        how="left"
    )
    .drop(columns=["posteam"])
    .merge(
        avg_start,
        left_on="defteam",
        right_on="posteam",
        how="left"
    )
    .drop(columns=["posteam"])
)

def_and_st.head()

,defteam,def_epa,special_teams_epa,avg_start
0,BUF,-0.010997,-33.418414,71.229167
1,DAL,0.167388,4.182480,73.287879
2,TB,0.013279,-25.591023,67.976471
3,CAR,0.070432,6.110069,70.448718
4,JAX,-0.086168,26.744811,66.914286


In [33]:
def_and_st["def_epa_grade"] = grade_metric(def_and_st, "def_epa", high_is_good=False)
def_and_st["special_teams_epa_grade"] = grade_metric(def_and_st, "special_teams_epa", high_is_good=True)
def_and_st["avg_start_grade"] = grade_metric(def_and_st, "avg_start", high_is_good=False)

def_and_st["def_and_st_grade"] = (
    def_and_st[["def_epa_grade", "special_teams_epa_grade", "avg_start_grade"]]
    .mean(axis=1)
)

def_and_st = def_and_st.sort_values("def_and_st_grade", ascending=False)

def_and_st.head()

,defteam,def_epa,special_teams_epa,avg_start,def_epa_grade,special_teams_epa_grade,avg_start_grade,def_and_st_grade
15,HOU,-0.160723,51.777008,66.520000,97.334576,95.076341,96.797166,96.402694
4,JAX,-0.086168,26.744811,66.914286,85.806162,80.330257,94.740986,86.959135
8,SEA,-0.123929,72.363005,69.314961,93.417323,98.953483,58.334296,83.568367
28,MIN,-0.099459,13.628391,69.067416,88.973051,66.818520,63.901346,73.230972
13,CHI,0.031369,16.810705,66.636364,38.768722,70.418002,96.274354,68.487026


## Quarterback Support Grade

In [34]:
qbs = (
    pass_pro[["posteam", "pass_pro_grade"]]
    .merge(
        play_calling[["posteam", "play_calling_grade"]],
        on="posteam",
        how="left"
    )
    .merge(
        pass_catching[["posteam", "pass_catching_grade"]],
        on="posteam",
        how="left"
    )
    .merge(
        run_game[["posteam", "run_game_grade"]],
        on="posteam",
        how="left"
    )
    .merge(
        def_and_st[["defteam", "def_and_st_grade"]],
        left_on="posteam",
        right_on="defteam",
        how="left"
    )
    .drop(columns=["defteam"])
)

qbs.head()

,posteam,pass_pro_grade,play_calling_grade,pass_catching_grade,run_game_grade,def_and_st_grade
0,PIT,89.267997,67.814114,66.861899,81.010485,63.530372
1,BUF,82.573184,76.134776,75.099374,96.451395,30.143135
2,CHI,81.356790,75.164406,45.802606,74.737791,68.487026
3,DEN,78.977454,56.308891,49.299302,50.028282,52.828610
4,WAS,75.207100,63.482212,37.811462,70.059505,47.334286


In [35]:
qbs["quarterback_support_grade"] = (
    (0.3 * qbs["pass_pro_grade"]) +
    (0.25 * qbs["play_calling_grade"]) +
    (0.2 * qbs["pass_catching_grade"]) +
    (0.15 * qbs["run_game_grade"]) +
    (0.1 * qbs["def_and_st_grade"])
)

qbs = qbs.sort_values("quarterback_support_grade", ascending=False)

qbs.head()

,posteam,pass_pro_grade,play_calling_grade,pass_catching_grade,run_game_grade,def_and_st_grade,quarterback_support_grade
1,BUF,82.573184,76.134776,75.099374,96.451395,30.143135,76.307547
0,PIT,89.267997,67.814114,66.861899,81.010485,63.530372,75.610917
2,CHI,81.356790,75.164406,45.802606,74.737791,68.487026,70.418031
9,LA,63.545474,61.402086,50.284073,88.942042,52.226363,63.034921
5,SEA,72.215685,36.479730,81.750178,41.006697,83.568367,61.642515


## Quaeterback Performance Grade

In [36]:
qb_pbp = (
    pbp
    .filter(pl.col("qb_dropback") == 1)
    .group_by("passer_player_id")
    .agg(
        pl.mean("epa").alias("epa_per_dropback"),
        pl.mean("success").alias("success_rate")
    )
).to_pandas()

qb_pbp.head()

,passer_player_id,epa_per_dropback,success_rate
0,00-0039910,-0.028901,0.441176
1,00-0039851,0.175148,0.505140
2,00-0033357,-0.166600,0.333333
3,00-0035289,-0.493437,0.230769
4,00-0036389,0.042972,0.427203


In [37]:
qb_overall = (
    ngs_qbs_filtered
    .merge(
        qb_pbp,
        left_on="player_gsis_id",
        right_on="passer_player_id",
        how="left"
    )
    .drop(columns=["passer_player_id", "player_gsis_id"])
)

qb_overall.head()

,player_short_name,team_abbr,attempts,passer_rating,completion_percentage_above_expectation,headshot,epa_per_dropback,success_rate
0,B.Nix,DEN,612,87.806373,-2.067418,https://static.www.nfl.com/image/upload/f_auto...,0.089747,0.440058
1,D.Prescott,DAL,600,99.527778,4.445367,https://static.www.nfl.com/image/upload/f_auto...,0.165174,0.477778
2,M.Stafford,LAR,597,109.195282,1.476147,https://static.www.nfl.com/image/upload/f_auto...,0.221393,0.518219
3,J.Goff,DET,578,105.485871,1.581453,https://static.www.nfl.com/image/upload/f_auto...,0.174478,0.483713
4,C.Williams,CHI,568,90.126174,-6.874683,https://static.www.nfl.com/image/upload/f_auto...,0.081919,0.435860


In [38]:
qb_overall["passer_rating_grade"] = grade_metric(qb_overall, "passer_rating", high_is_good=True)
qb_overall["cpoe_grade"] = grade_metric(qb_overall, "completion_percentage_above_expectation", high_is_good=True)
qb_overall["epa_per_dropback_grade"] = grade_metric(qb_overall, "epa_per_dropback", high_is_good=True)
qb_overall["success_rate_grade"] = grade_metric(qb_overall, "success_rate", high_is_good=True)

qb_overall.head()

,player_short_name,team_abbr,attempts,passer_rating,completion_percentage_above_expectation,headshot,epa_per_dropback,success_rate,passer_rating_grade,cpoe_grade,epa_per_dropback_grade,success_rate_grade
0,B.Nix,DEN,612,87.806373,-2.067418,https://static.www.nfl.com/image/upload/f_auto...,0.089747,0.440058,32.683860,24.672131,72.123107,42.684484
1,D.Prescott,DAL,600,99.527778,4.445367,https://static.www.nfl.com/image/upload/f_auto...,0.165174,0.477778,79.204267,90.028409,88.314119,74.343565
2,M.Stafford,LAR,597,109.195282,1.476147,https://static.www.nfl.com/image/upload/f_auto...,0.221393,0.518219,96.816969,65.022998,94.962978,93.977076
3,J.Goff,DET,578,105.485871,1.581453,https://static.www.nfl.com/image/upload/f_auto...,0.174478,0.483713,92.718073,66.193960,89.713220,78.403808
4,C.Williams,CHI,568,90.126174,-6.874683,https://static.www.nfl.com/image/upload/f_auto...,0.081919,0.435860,42.118695,1.627909,69.978385,39.060943


In [39]:
qbp = qb_overall[["player_short_name", "team_abbr", "headshot", "attempts", "epa_per_dropback_grade", "success_rate_grade", "cpoe_grade", "passer_rating_grade"]]

qbp["quarterback_performance_grade"] = (
    (0.4 * qbp["epa_per_dropback_grade"]) +
    (0.4 * qbp["success_rate_grade"]) +
    (0.15 * qbp["cpoe_grade"]) +
    (0.05 * qbp["passer_rating_grade"])
)

qbp = qbp.sort_values("quarterback_performance_grade", ascending=False)

qbp.loc[qbp["team_abbr"] == "LAR", ["team_abbr"]] = "LA"

qbp.head()

/var/folders/_v/f9jlhvnd2xg9kybwfc9yh6nw0000gn/T/ipykernel_68660/558094566.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  qbp["quarterback_performance_grade"] = (


,player_short_name,team_abbr,headshot,attempts,epa_per_dropback_grade,success_rate_grade,cpoe_grade,passer_rating_grade,quarterback_performance_grade
11,D.Maye,NE,https://static.www.nfl.com/image/upload/f_auto...,492,89.809120,89.655125,99.654691,98.972618,91.682533
2,M.Stafford,LA,https://static.www.nfl.com/image/upload/f_auto...,597,94.962978,93.977076,65.022998,96.816969,90.170320
18,J.Love,GB,https://static.www.nfl.com/image/upload/f_auto...,439,97.107718,83.469632,86.281427,83.930941,89.369701
14,S.Darnold,SEA,https://static.www.nfl.com/image/upload/f_auto...,477,84.799620,92.830939,89.274514,77.902550,88.338529
27,B.Purdy,SF,https://static.www.nfl.com/image/upload/f_auto...,284,81.201788,93.168148,92.955371,82.030139,87.792787


## Quarterback Performance vs Support

In [40]:
support_and_performance = (
    qbp[["player_short_name", "team_abbr", "headshot", "quarterback_performance_grade"]]
    .merge(
        qbs[["posteam", "quarterback_support_grade"]],
        left_on=["team_abbr"],
        right_on=["posteam"],
        how="left"
    )
    .merge(
        teams_pd[["team_abbr", "team_color", "team_logo_espn"]],
        left_on="team_abbr",
        right_on="team_abbr",
        how="left"
    )
    .drop(columns=["posteam"])
)

support_and_performance.head()

,player_short_name,team_abbr,headshot,quarterback_performance_grade,quarterback_support_grade,team_color,team_logo_espn
0,D.Maye,NE,https://static.www.nfl.com/image/upload/f_auto...,91.682533,40.981994,#002244,https://a.espncdn.com/i/teamlogos/nfl/500/ne.png
1,M.Stafford,LA,https://static.www.nfl.com/image/upload/f_auto...,90.170320,63.034921,#003594,https://a.espncdn.com/i/teamlogos/nfl/500/lar.png
2,J.Love,GB,https://static.www.nfl.com/image/upload/f_auto...,89.369701,55.830646,#203731,https://a.espncdn.com/i/teamlogos/nfl/500/gb.png
3,S.Darnold,SEA,https://static.www.nfl.com/image/upload/f_auto...,88.338529,61.642515,#002244,https://a.espncdn.com/i/teamlogos/nfl/500/sea.png
4,B.Purdy,SF,https://static.www.nfl.com/image/upload/f_auto...,87.792787,50.591836,#AA0000,https://a.espncdn.com/i/teamlogos/nfl/500/sf.png
